In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into memory (variables are reused if rerun in the same kernel)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error_logs = pd.read_csv('error_logs.csv')

# Helper to ensure timestamp column exists and convert to timezone-aware datetimes (UTC->Asia/Shanghai)
def to_shanghai_dt(series):
    # make sure integer
    s = pd.to_datetime(series.astype('int64'), unit='s', utc=True)
    return s.dt.tz_convert(tz)

# Summaries for metric.csv
metric_rows = len(df_metric)
metric_ts = to_shanghai_dt(df_metric['timestamp']) if 'timestamp' in df_metric else pd.Series([], dtype='datetime64[ns]')
metric_min_ts = metric_ts.min() if not metric_ts.empty else pd.NaT
metric_max_ts = metric_ts.max() if not metric_ts.empty else pd.NaT
metric_unique_cmdb = pd.Series(df_metric['cmdb_id'].unique()) if 'cmdb_id' in df_metric else pd.Series([], dtype=object)
metric_unique_cmdb_count = metric_unique_cmdb.shape[0]
metric_unique_cmdb_list = metric_unique_cmdb.head(50).reset_index(drop=True)  # up to 50

# top kpi_name counts (up to 50)
if 'kpi_name' in df_metric:
    metric_kpi_counts = df_metric['kpi_name'].value_counts().head(50).reset_index()
    metric_kpi_counts.columns = ['kpi_name', 'count']
else:
    metric_kpi_counts = pd.DataFrame(columns=['kpi_name', 'count'])

metric_overview = pd.DataFrame({
    'rows':[metric_rows],
    'min_timestamp':[metric_min_ts],
    'max_timestamp':[metric_max_ts],
    'unique_cmdb_count':[metric_unique_cmdb_count],
    'unique_cmdb_list':[', '.join(metric_unique_cmdb_list.astype(str).tolist())]
})

# Summaries for trace.csv
trace_rows = len(df_trace)
trace_ts = to_shanghai_dt(df_trace['timestamp']) if 'timestamp' in df_trace else pd.Series([], dtype='datetime64[ns]')
trace_min_ts = trace_ts.min() if not trace_ts.empty else pd.NaT
trace_max_ts = trace_ts.max() if not trace_ts.empty else pd.NaT
trace_unique_cmdb = pd.Series(df_trace['cmdb_id'].unique()) if 'cmdb_id' in df_trace else pd.Series([], dtype=object)
trace_unique_cmdb_count = trace_unique_cmdb.shape[0]
trace_unique_cmdb_list = trace_unique_cmdb.head(50).reset_index(drop=True)

# top trace_name counts (up to 50)
if 'trace_name' in df_trace:
    trace_name_counts = df_trace['trace_name'].value_counts().head(50).reset_index()
    trace_name_counts.columns = ['trace_name', 'count']
else:
    trace_name_counts = pd.DataFrame(columns=['trace_name', 'count'])

trace_overview = pd.DataFrame({
    'rows':[trace_rows],
    'min_timestamp':[trace_min_ts],
    'max_timestamp':[trace_max_ts],
    'unique_cmdb_count':[trace_unique_cmdb_count],
    'unique_cmdb_list':[', '.join(trace_unique_cmdb_list.astype(str).tolist())]
})

# Summaries for log.csv
log_rows = len(df_log)
log_ts = to_shanghai_dt(df_log['timestamp']) if 'timestamp' in df_log else pd.Series([], dtype='datetime64[ns]')
log_min_ts = log_ts.min() if not log_ts.empty else pd.NaT
log_max_ts = log_ts.max() if not log_ts.empty else pd.NaT
log_unique_cmdb = pd.Series(df_log['cmdb_id'].unique()) if 'cmdb_id' in df_log else pd.Series([], dtype=object)
log_unique_cmdb_count = log_unique_cmdb.shape[0]
log_unique_cmdb_list = log_unique_cmdb.head(50).reset_index(drop=True)

# top log_name counts (up to 50)
if 'log_name' in df_log:
    log_name_counts = df_log['log_name'].value_counts().head(50).reset_index()
    log_name_counts.columns = ['log_name', 'count']
else:
    log_name_counts = pd.DataFrame(columns=['log_name', 'count'])

log_overview = pd.DataFrame({
    'rows':[log_rows],
    'min_timestamp':[log_min_ts],
    'max_timestamp':[log_max_ts],
    'unique_cmdb_count':[log_unique_cmdb_count],
    'unique_cmdb_list':[', '.join(log_unique_cmdb_list.astype(str).tolist())]
})

# Summaries for error_logs.csv (top 20 rows sample)
err_rows = len(df_error_logs)
err_ts = to_shanghai_dt(df_error_logs['timestamp']) if 'timestamp' in df_error_logs else pd.Series([], dtype='datetime64[ns]')
err_min_ts = err_ts.min() if not err_ts.empty else pd.NaT
err_max_ts = err_ts.max() if not err_ts.empty else pd.NaT
err_unique_cmdb = pd.Series(df_error_logs['cmdb_id'].unique()) if 'cmdb_id' in df_error_logs else pd.Series([], dtype=object)
err_unique_cmdb_count = err_unique_cmdb.shape[0]
err_unique_cmdb_list = err_unique_cmdb.head(50).reset_index(drop=True)

error_overview = pd.DataFrame({
    'rows':[err_rows],
    'min_timestamp':[err_min_ts],
    'max_timestamp':[err_max_ts],
    'unique_cmdb_count':[err_unique_cmdb_count],
    'unique_cmdb_list':[', '.join(err_unique_cmdb_list.astype(str).tolist())]
})

# Prepare top 20 sample (timestamp converted to Asia/Shanghai string for compactness)
if {'timestamp','cmdb_id','message'}.issubset(df_error_logs.columns):
    error_sample = df_error_logs[['timestamp','cmdb_id','message']].copy().head(20)
    error_sample['timestamp'] = to_shanghai_dt(error_sample['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S %z')
else:
    # fallback: show whatever columns exist up to 20 rows
    error_sample = df_error_logs.head(20)[['timestamp','cmdb_id','message']].copy() if 'timestamp' in df_error_logs.columns else df_error_logs.head(20)

# Final compact outputs (limited lists/counts as requested)
metric_overview, metric_kpi_counts, trace_overview, trace_name_counts, log_overview, log_name_counts, error_overview, error_sample

```
Out[1]:
```
```python
# Compose a compact plain-English summary from previously loaded summary variables.
# Assumes metric_overview, metric_kpi_counts, trace_overview, trace_name_counts, log_overview, log_name_counts, error_overview, error_sample exist in kernel.

def fmt_ts(val):
    try:
        return pd.to_datetime(val).strftime('%Y-%m-%d %H:%M %z')
    except Exception:
        try:
            return str(val)
        except:
            return 'N/A'

# metric.csv overview
m_row = int(metric_overview.at[0, 'rows'])
m_min = metric_overview.at[0, 'min_timestamp']
m_max = metric_overview.at[0, 'max_timestamp']
m_cmdb_count = int(metric_overview.at[0, 'unique_cmdb_count'])
m_cmdb_list = metric_overview.at[0, 'unique_cmdb_list']

top_kpis = []
for _, r in metric_kpi_counts.head(10).iterrows():
    top_kpis.append(f"{r['kpi_name']} ({int(r['count'])})")
top_kpis_txt = "; ".join(top_kpis) if top_kpis else "none"

# trace.csv overview
t_row = int(trace_overview.at[0, 'rows'])
t_min = trace_overview.at[0, 'min_timestamp']
t_max = trace_overview.at[0, 'max_timestamp']
t_cmdb_count = int(trace_overview.at[0, 'unique_cmdb_count'])
t_cmdb_list = trace_overview.at[0, 'unique_cmdb_list']

top_traces = []
for _, r in trace_name_counts.head(10).iterrows():
    top_traces.append(f"{r['trace_name']} ({int(r['count'])})")
top_traces_txt = "; ".join(top_traces) if top_traces else "none"

# log.csv overview
l_row = int(log_overview.at[0, 'rows'])
l_min = log_overview.at[0, 'min_timestamp']
l_max = log_overview.at[0, 'max_timestamp']
l_cmdb_count = int(log_overview.at[0, 'unique_cmdb_count'])
l_cmdb_list = log_overview.at[0, 'unique_cmdb_list']

top_logs = []
for _, r in log_name_counts.head(10).iterrows():
    top_logs.append(f"{r['log_name']} ({int(r['count'])})")
top_logs_txt = "; ".join(top_logs) if top_logs else "none"

# error_logs.csv overview
e_row = int(error_overview.at[0, 'rows'])
e_min = error_overview.at[0, 'min_timestamp']
e_max = error_overview.at[0, 'max_timestamp']

# Build summary string (compact)
summary = (
    "Summary of loaded telemetry files:\n\n"
    f"metric.csv: rows={m_row}; time_range={fmt_ts(m_min)} to {fmt_ts(m_max)}; unique_cmdb_id={m_cmdb_count}; "
    f"cmdb_examples=[{m_cmdb_list}] ; top_kpi_names (top 10)=[{top_kpis_txt}]\n\n"
    f"trace.csv: rows={t_row}; time_range={fmt_ts(t_min)} to {fmt_ts(t_max)}; unique_cmdb_id={t_cmdb_count}; "
    f"cmdb_examples=[{t_cmdb_list}] ; top_trace_names (top 10)=[{top_traces_txt}]\n\n"
    f"log.csv: rows={l_row}; time_range={fmt_ts(l_min)} to {fmt_ts(l_max)}; unique_cmdb_id={l_cmdb_count}; "
    f"cmdb_examples=[{l_cmdb_list}] ; top_log_names (top 10)=[{top_logs_txt}]\n\n"
    f"error_logs.csv: rows={e_row}; time_range={fmt_ts(e_min)} to {fmt_ts(e_max)}; sample_rows_shown=0 (no entries)\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     rows             min_timestamp             max_timestamp  unique_cmdb_count                                   unique_cmdb_list
0  322439 2022-03-21 08:30:00+08:00 2022-03-21 09:30:00+08:00                 59  adservice, adservice-0, adservice-1, adservice...,                                              kpi_name  count
0   container.node-6.container_network_transmit_er...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_transmit_pa...   2196
3   container.node-6.container_network_receive_MB....   2196
4   container.node-6.container_network_receive_err...   2196
..                                                ...    ...
45  container.node-6.container_fs_sector_reads./de...   2074
46    container.node-6.container_tasks_state.sleeping   2074
47   container.node-6.container_memory_working_set_MB   2074
48  container.node-6.container_fs_reads_merged./de...   2074
49    container.node-6.container_fs_reads_MB./dev/vda   2074

[50 rows x 2 columns],     rows             min_timestamp             max_timestamp  unique_cmdb_count                                   unique_cmdb_list
0  27656 2022-03-21 09:00:00+08:00 2022-03-21 09:29:00+08:00                 36  adservice-0, adservice-1, adservice-2, cartser...,                                   trace_name  count
0        trace.from_frontend-0.duration_mean    598
1         trace.from_frontend-0.duration_p95    598
2           trace.from_frontend-0.error_rate    598
3            trace.from_frontend-0.row_count    598
4        trace.from_frontend-1.duration_mean    595
..                                       ...    ...
45     trace.to_currencyservice-2.error_rate    173
46   trace.to_currencyservice-2.duration_p95    173
47  trace.to_currencyservice-2.duration_mean    173
48  trace.to_currencyservice-1.duration_mean    172
49   trace.to_currencyservice-1.duration_p95    172

[50 rows x 2 columns],    rows             min_timestamp             max_timestamp  unique_cmdb_count                                   unique_cmdb_list
0  1822 2022-03-21 09:00:00+08:00 2022-03-21 09:30:00+08:00                 31  adservice-0, adservice-1, adservice-2, cartser...,           log_name  count
0  log.error_count    911
1    log.row_count    911,    rows min_timestamp max_timestamp  unique_cmdb_count unique_cmdb_list
0     0           NaT           NaT                  0                 , Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column and a timezone-aware datetime column for Asia/Shanghai
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)
df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(tz)

# 1) Global P95 threshold and full series count per (cmdb_id, kpi_name)
df_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(full_series_count='count', p95_threshold=lambda x: x.quantile(0.95))
)

# 2) Filter to incident window 2022-03-21 09:00:00 to 2022-03-21 09:30:00 (Asia/Shanghai)
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
mask_window = (df_metric['ts_sh'] >= start_sh) & (df_metric['ts_sh'] <= end_sh)
df_window = df_metric[mask_window].copy()

# Merge window rows with global thresholds
df_window_merged = df_window.merge(df_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark breaches where value >= global P95 (use >= as requested)
df_window_merged['is_breach'] = df_window_merged['value'] >= df_window_merged['p95_threshold']

# Aggregations per pair for in-window stats
agg_window = df_window_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    in_window_count=('value', 'count'),
    breaches_count=('is_breach', 'sum'),
    max_value_in_window=('value', 'max')
)

# Earliest breach timestamp per pair (Asia/Shanghai formatted) - may be missing
df_breaches = df_window_merged[df_window_merged['is_breach']].copy()
if not df_breaches.empty:
    earliest = (
        df_breaches.groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_breach_ts_sh=('ts_sh', 'min'))
    )
    earliest['earliest_breach_timestamp'] = earliest['earliest_breach_ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    earliest = earliest.drop(columns=['earliest_breach_ts_sh'])
else:
    earliest = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'earliest_breach_timestamp'])

# Combine thresholds + window aggregates + earliest breach
df_result = (
    df_thresholds
    .merge(agg_window, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for pairs that had no data in-window
df_result['in_window_count'] = df_result['in_window_count'].fillna(0).astype(int)
df_result['breaches_count'] = df_result['breaches_count'].fillna(0).astype(int)
df_result['max_value_in_window'] = df_result['max_value_in_window'].astype(float)
df_result['earliest_breach_timestamp'] = df_result['earliest_breach_timestamp'].fillna('')

# 3) Select top 20 by breaches_count desc; if no breaches at all, sort by max_value_in_window desc
total_breaches = int(df_result['breaches_count'].sum())
if total_breaches > 0:
    df_top = df_result.sort_values(['breaches_count', 'max_value_in_window'], ascending=[False, False]).head(20)
else:
    df_top = df_result.sort_values('max_value_in_window', ascending=False).head(20)

# Select and order required columns
df_top = df_top[[
    'cmdb_id',
    'kpi_name',
    'full_series_count',
    'p95_threshold',
    'in_window_count',
    'breaches_count',
    'earliest_breach_timestamp',
    'max_value_in_window'
]]

# Keep intermediate tables in kernel: df_thresholds, df_window, df_window_merged, df_breaches, earliest, df_result
df_thresholds.shape, df_window.shape, df_window_merged.shape, df_breaches.shape, df_top.shape, df_top.head(20)

```
Out[2]:
```
```python
# Compose a compact plain-English summary based on existing intermediate tables:
# Requires df_thresholds, df_result, df_top to exist in the kernel (created previously).

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
window_str = "2022-03-21 09:00:00 to 2022-03-21 09:30:00 (Asia/Shanghai)"

# Basic counts
total_pairs = int(df_thresholds.shape[0]) if 'df_thresholds' in globals() else None
pairs_with_breaches = int((df_result['breaches_count'] > 0).sum())
total_breaches = int(df_result['breaches_count'].sum())

# Characteristics of top results
top_count_unique = sorted(df_top['breaches_count'].unique().tolist())
top_in_window_counts = sorted(df_top['in_window_count'].unique().tolist())
top_full_series_counts = sorted(df_top['full_series_count'].unique().tolist())

# Sample KPI names appearing in top 20
sample_kpis = df_top['kpi_name'].unique()[:10].tolist()

# Earliest breach times in top 20 (unique)
earliest_breach_times = sorted([t for t in df_top['earliest_breach_timestamp'].unique() if t != ''])

summary = (
    f"Metric.csv analysis for window {window_str} (UTC+8):\n\n"
    f"- Total unique (cmdb_id, kpi_name) pairs analyzed: {total_pairs}\n"
    f"- Pairs with >=1 breach in-window: {pairs_with_breaches}; total breach events summed across pairs: {total_breaches}\n\n"
    f"- Top-20 pairs (sorted by breaches) summary:\n"
    f"  * breaches_count values in top-20: {top_count_unique}\n"
    f"  * in_window_count values in top-20: {top_in_window_counts}\n"
    f"  * full_series_count values in top-20: {top_full_series_counts}\n"
    f"  * earliest breach times in top-20 (sample): {earliest_breach_times[:3]}\n"
    f"  * sample KPI names among top-20: {sample_kpis}\n\n"
    "Observed pattern / conclusion:\n"
    "- The top 20 pairs all show repeated breaches across the entire 31-minute window (breaches_count = in_window_count = 31),\n"
    "  and full_series_count for these pairs is 61. The earliest breach timestamps are at the window start (2022-03-21 09:00:00 +0800).\n"
    "- Many top KPIs are timestamp-like or start-time metrics (e.g., runtime/java start times, container_start_time_seconds)\n"
    "  and some are node-level 'system.disk.total' or large memory metrics. Their p95 thresholds are large constant numbers\n"
    "  (timestamps or large capacity values) and the window values equal/ exceed those thresholds, producing breaches.\n\n"
    "Actionable note:\n"
    "- These breaches appear driven by static or monotonic metrics (start times, static totals) rather than transient resource spikes.\n"
    "- Recommend prioritizing investigation on runtime/start-time KPIs and confirming whether such KPIs should be used for anomaly detection,\n"
    "  and to inspect node.disk.total entries for correctness (units/labels) before treating them as true runtime anomalies.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((5298, 4), (163859, 6), (163859, 9), (121202, 9), (20, 8),                       cmdb_id                                           kpi_name  full_series_count  p95_threshold  in_window_count  breaches_count  earliest_breach_timestamp  max_value_in_window
857                adservice2                runtime.java_lang_Runtime_StartTime                 61   1.647819e+12               31              31  2022-03-21 09:00:00 +0800         1.647819e+12
250                 adservice                runtime.java_lang_Runtime_StartTime                 61   1.647144e+12               31              31  2022-03-21 09:00:00 +0800         1.647144e+12
3684                   node-5                                  system.disk.total                 61   1.854945e+10               31              31  2022-03-21 09:00:00 +0800         1.854945e+10
3450                   node-1                                  system.disk.total                 61   1.406688e+10               31              31  2022-03-21 09:00:00 +0800         1.406688e+10
3509                   node-2                                  system.disk.total                 61   1.406688e+10               31              31  2022-03-21 09:00:00 +0800         1.406688e+10
3567                   node-3                                  system.disk.total                 61   1.406688e+10               31              31  2022-03-21 09:00:00 +0800         1.406688e+10
238                 adservice  runtime.java_lang_OperatingSystem_CommittedVir...                 61   3.591926e+09               31              31  2022-03-21 09:00:00 +0800         3.591926e+09
845                adservice2  runtime.java_lang_OperatingSystem_CommittedVir...                 61   3.000410e+09               31              31  2022-03-21 09:00:00 +0800         3.360096e+09
1003             adservice2-0      container.node-6.container_start_time_seconds                 61   1.647819e+09               31              31  2022-03-21 09:00:00 +0800         1.647819e+09
1405           cartservice2-0      container.node-5.container_start_time_seconds                 61   1.647782e+09               31              31  2022-03-21 09:00:00 +0800         1.647782e+09
1109            cartservice-0      container.node-6.container_start_time_seconds                 61   1.647760e+09               31              31  2022-03-21 09:00:00 +0800         1.647760e+09
1305            cartservice-2      container.node-6.container_start_time_seconds                 61   1.647759e+09               31              31  2022-03-21 09:00:00 +0800         1.647759e+09
1209            cartservice-1      container.node-6.container_start_time_seconds                 61   1.647757e+09               31              31  2022-03-21 09:00:00 +0800         1.647757e+09
3993         paymentservice-2      container.node-6.container_start_time_seconds                 61   1.647318e+09               31              31  2022-03-21 09:00:00 +0800         1.647318e+09
2333       currencyservice2-0      container.node-6.container_start_time_seconds                 61   1.647236e+09               31              31  2022-03-21 09:00:00 +0800         1.647236e+09
2245        currencyservice-2      container.node-6.container_start_time_seconds                 61   1.647236e+09               31              31  2022-03-21 09:00:00 +0800         1.647236e+09
3109               frontend-2      container.node-5.container_start_time_seconds                 61   1.647144e+09               31              31  2022-03-21 09:00:00 +0800         1.647144e+09
4801  recommendationservice-2      container.node-6.container_start_time_seconds                 61   1.647144e+09               31              31  2022-03-21 09:00:00 +0800         1.647144e+09
1793        checkoutservice-2      container.node-5.container_start_time_seconds                 61   1.647144e+09               31              31  2022-03-21 09:00:00 +0800         1.647144e+09
2921               frontend-1      container.node-5.container_start_time_seconds                 61   1.647144e+09               31              31  2022-03-21 09:00:00 +0800         1.647144e+09)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import re

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware timestamp in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(tz)

# Substrings to match (case-insensitive)
substrs = [
    "cpu", "memory", "mem", "network", "transmit", "receive", "retransmit",
    "packet", "error", "drop", "latency", "delay", "disk", "io", "read",
    "write", "process", "tasks", "heap", "gc"
]
# Build regex pattern (word fragments allowed)
pattern = '|'.join(re.escape(s) for s in substrs)

# Filter metric rows where kpi_name contains any of the substrings (case-insensitive)
mask_match = df_metric['kpi_name'].str.contains(pattern, case=False, na=False)
df_matched = df_metric[mask_match].copy()

# 1) Compute global P95 and full_series_count for each (cmdb_id, kpi_name) using ALL rows for that pair
df_thresholds_kpi = (
    df_matched
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(full_series_count='count', p95_threshold=lambda x: x.quantile(0.95))
)

# 2) Filter rows within the incident window (Asia/Shanghai)
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
mask_window = (df_matched['ts_sh'] >= start_sh) & (df_matched['ts_sh'] <= end_sh)
df_window_matched = df_matched[mask_window].copy()

# Merge window rows with thresholds to evaluate breaches
df_window_merged = df_window_matched.merge(df_thresholds_kpi, on=['cmdb_id', 'kpi_name'], how='left')

# Mark breaches where value >= global P95 threshold (use >= as requested)
df_window_merged['is_breach'] = df_window_merged['value'] >= df_window_merged['p95_threshold']

# Aggregations per pair for in-window stats
agg_window = df_window_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    in_window_count=('value', 'count'),
    breaches_count=('is_breach', 'sum'),
    max_value_in_window=('value', 'max')
)

# Earliest breach timestamp per pair (formatted)
df_breaches = df_window_merged[df_window_merged['is_breach']].copy()
if not df_breaches.empty:
    earliest = (
        df_breaches.groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_breach_ts_sh=('ts_sh', 'min'))
    )
    earliest['earliest_breach_timestamp'] = earliest['earliest_breach_ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    earliest = earliest.drop(columns=['earliest_breach_ts_sh'])
else:
    earliest = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'earliest_breach_timestamp'])

# Combine thresholds with window aggregates, include pairs with zero in-window data
df_kpi_result = (
    df_thresholds_kpi
    .merge(agg_window, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for pairs missing window data
df_kpi_result['in_window_count'] = df_kpi_result['in_window_count'].fillna(0).astype(int)
df_kpi_result['breaches_count'] = df_kpi_result['breaches_count'].fillna(0).astype(int)
df_kpi_result['max_value_in_window'] = df_kpi_result['max_value_in_window'].astype(float)
df_kpi_result['earliest_breach_timestamp'] = df_kpi_result['earliest_breach_timestamp'].fillna('')

# 3) Return top 20 rows sorted by breaches_count desc then max_value_in_window desc
df_top20 = df_kpi_result.sort_values(['breaches_count', 'max_value_in_window'], ascending=[False, False]).head(20)

# Also return distinct matched kpi_name list (up to 50) and count of distinct cmdb_id matched
matched_kpi_names = pd.DataFrame({'kpi_name': df_matched['kpi_name'].value_counts().index.tolist()[:50]})
matched_cmdb_count = int(df_matched['cmdb_id'].nunique())

# Keep intermediate variables in kernel as requested:
# df_matched, df_thresholds_kpi, df_window_matched, df_window_merged, df_breaches, df_kpi_result

df_top20, matched_kpi_names, matched_cmdb_count

```
Out[3]:
```
```python
# Compose a compact plain-English summary of the KPI scan results (uses existing kernel variables).
try:
    distinct_kpi_count = int(df_matched['kpi_name'].nunique())
    distinct_cmdb_count = int(df_matched['cmdb_id'].nunique())
    top_table = df_top20.copy()
except Exception as e:
    summary = f"Error: required intermediate variables not found in kernel: {e}"
    summary

# Build concise summary
summary_lines = [
    f"Scan of metric.csv for health-related KPIs (substr match).",
    f"- Distinct matched kpi_name count: {distinct_kpi_count}",
    f"- Distinct matched cmdb_id count: {distinct_cmdb_count}",
    "",
    "Top results (up to 20 rows) summary:",
]

# If top_table empty
if top_table.shape[0] == 0:
    summary_lines.append("- No matched KPI pairs had any in-window data or breaches.")
else:
    # Check common pattern among top rows
    breaches_vals = sorted(top_table['breaches_count'].unique())
    in_window_vals = sorted(top_table['in_window_count'].unique())
    full_counts = sorted(top_table['full_series_count'].unique())
    earliest_times = [t for t in top_table['earliest_breach_timestamp'].unique() if t]
    sample_kpis = top_table['kpi_name'].unique()[:6].tolist()
    summary_lines.append(f"- Top-20 pairs: breaches_count values = {breaches_vals}; in_window_count values = {in_window_vals}; full_series_count values = {full_counts}")
    summary_lines.append(f"- Earliest breach times seen (sample): {earliest_times[:3]}")
    summary_lines.append(f"- Sample KPI names in top-20: {sample_kpis}")
    summary_lines.append("")
    summary_lines.append("Interpretation:")
    summary_lines.append("- Many top pairs show breaches for every minute in the 09:00–09:30 window (breaches_count = in_window_count = 31),")
    summary_lines.append("  and their full_series_count is 61 with earliest breach at 2022-03-21 09:00:00 +0800.")
    summary_lines.append("- The top KPIs are largely static/monotonic metrics (e.g., disk totals, container/start-time timestamps, JVM memory/GC info).")
    summary_lines.append("- These appear to be constant or timestamp-like values exceeding their p95 (which are large constants), so breaches likely reflect metric semantics rather than transient resource problems.")
    summary_lines.append("")
    summary_lines.append("Actionable recommendation:")
    summary_lines.append("- Exclude static/timestamp KPIs (start times, total capacities) from anomaly detection or adjust thresholds/aggregation.")
    summary_lines.append("- Focus investigation on genuine dynamic health KPIs (CPU, memory usage, network error/packet rates, I/O rates) that show transient spikes.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  full_series_count  p95_threshold  in_window_count  breaches_count  max_value_in_window  earliest_breach_timestamp
3293      node-5                                  system.disk.total                 61   1.854945e+10               31              31         1.854945e+10  2022-03-21 09:00:00 +0800
3161      node-1                                  system.disk.total                 61   1.406688e+10               31              31         1.406688e+10  2022-03-21 09:00:00 +0800
3194      node-2                                  system.disk.total                 61   1.406688e+10               31              31         1.406688e+10  2022-03-21 09:00:00 +0800
3227      node-3                                  system.disk.total                 61   1.406688e+10               31              31         1.406688e+10  2022-03-21 09:00:00 +0800
224    adservice  runtime.java_lang_OperatingSystem_CommittedVir...                 61   3.591926e+09               31              31         3.591926e+09  2022-03-21 09:00:00 +0800
782   adservice2  runtime.java_lang_OperatingSystem_CommittedVir...                 61   3.000410e+09               31              31         3.360096e+09  2022-03-21 09:00:00 +0800
40     adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
41     adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
88     adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
89     adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
154    adservice  runtime.java_lang_MemoryPool_PeakUsage_max.Com...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
196    adservice  runtime.java_lang_MemoryPool_Usage_max.Compres...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
598   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
599   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
646   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
647   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
712   adservice2  runtime.java_lang_MemoryPool_PeakUsage_max.Com...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
754   adservice2  runtime.java_lang_MemoryPool_Usage_max.Compres...                 61   1.073742e+09               31              31         1.073742e+09  2022-03-21 09:00:00 +0800
229    adservice  runtime.java_lang_OperatingSystem_TotalPhysica...                 61   6.291456e+08               31              31         6.291456e+08  2022-03-21 09:00:00 +0800
11     adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61   5.392454e+08               31              31         5.392454e+08  2022-03-21 09:00:00 +0800,                                              kpi_name
0   container.node-6.container_network_receive_err...
1   container.node-6.container_network_receive_pac...
2   container.node-6.container_network_receive_pac...
3   container.node-6.container_network_transmit_MB...
4   container.node-6.container_network_transmit_er...
..                                                ...
45         container.node-6.container_spec_cpu_shares
46  container.node-6.container_fs_io_time_seconds....
47  container.node-6.container_fs_io_current./dev/...
48        container.node-6.container_cpu_user_seconds
49       container.node-6.container_cpu_usage_seconds

[50 rows x 1 columns], 50)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import re

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware timestamp column for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(tz)

# Inclusion and exclusion substrings (case-insensitive)
includes = [
    "cpu", "usage", "user_seconds", "system_seconds", "mem", "memory", "rss",
    "working_set", "heap", "gc", "network", "transmit", "receive", "retransmit",
    "packet", "err", "drop", "latency", "delay", "fs_reads", "fs_writes",
    "io", "read", "write", "disk", "io_time", "tasks", "process"
]
excludes = [
    "start_time", "starttime", "runtime.java_lang", "garbagecollector",
    "memorypool", "system.disk.total", "committedvirtual", "totalphysical"
]

# Build regex patterns
inc_pattern = '(' + '|'.join(re.escape(s) for s in includes) + ')'
exc_pattern = '(' + '|'.join(re.escape(s) for s in excludes) + ')'

# Filter kpi_name by inclusion and exclusion (case-insensitive)
mask_inc = df_metric['kpi_name'].str.contains(inc_pattern, case=False, na=False)
mask_exc = df_metric['kpi_name'].str.contains(exc_pattern, case=False, na=False)
mask_selected = mask_inc & (~mask_exc)

df_selected = df_metric[mask_selected].copy()

# 1) Compute global P95 and full_series_count per (cmdb_id, kpi_name) using ALL rows for that pair
df_thresholds_sel = (
    df_selected
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(full_series_count='count', p95_threshold=lambda x: x.quantile(0.95))
)

# 2) Filter to incident window 2022-03-21 09:00:00 to 2022-03-21 09:30:00 (Asia/Shanghai)
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
mask_window_sel = (df_selected['ts_sh'] >= start_sh) & (df_selected['ts_sh'] <= end_sh)
df_window_sel = df_selected[mask_window_sel].copy()

# Merge window rows with global thresholds to evaluate breaches
df_window_merged_sel = df_window_sel.merge(df_thresholds_sel, on=['cmdb_id', 'kpi_name'], how='left')

# Mark breaches where value >= global P95 threshold
df_window_merged_sel['is_breach'] = df_window_merged_sel['value'] >= df_window_merged_sel['p95_threshold']

# Aggregations per pair for in-window stats
agg_window_sel = df_window_merged_sel.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    in_window_count=('value', 'count'),
    breaches_count=('is_breach', 'sum'),
    max_value_in_window=('value', 'max')
)

# Earliest breach timestamp per pair (formatted)
df_breaches_sel = df_window_merged_sel[df_window_merged_sel['is_breach']].copy()
if not df_breaches_sel.empty:
    earliest_sel = (
        df_breaches_sel.groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_breach_ts_sh=('ts_sh', 'min'))
    )
    earliest_sel['earliest_breach_timestamp'] = earliest_sel['earliest_breach_ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    earliest_sel = earliest_sel.drop(columns=['earliest_breach_ts_sh'])
else:
    earliest_sel = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'earliest_breach_timestamp'])

# Combine thresholds with window aggregates, include pairs with zero in-window data
df_result_sel = (
    df_thresholds_sel
    .merge(agg_window_sel, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_sel, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for pairs missing window data
df_result_sel['in_window_count'] = df_result_sel['in_window_count'].fillna(0).astype(int)
df_result_sel['breaches_count'] = df_result_sel['breaches_count'].fillna(0).astype(int)
df_result_sel['max_value_in_window'] = df_result_sel['max_value_in_window'].astype(float)
df_result_sel['earliest_breach_timestamp'] = df_result_sel['earliest_breach_timestamp'].fillna('')

# 3) Top 20 rows sorted by breaches_count desc then max_value_in_window desc
df_top20_sel = df_result_sel.sort_values(['breaches_count', 'max_value_in_window'], ascending=[False, False]).head(20)

# Distinct matched kpi_name list (up to 50) and count of distinct matched cmdb_id
matched_kpi_names_sel = pd.DataFrame({'kpi_name': df_selected['kpi_name'].value_counts().index.tolist()[:50]})
matched_cmdb_count_sel = int(df_selected['cmdb_id'].nunique())

# Keep intermediate variables in kernel:
# df_selected, df_thresholds_sel, df_window_sel, df_window_merged_sel, df_breaches_sel, df_result_sel, df_top20_sel

df_top20_sel, matched_kpi_names_sel, matched_cmdb_count_sel

```
Out[4]:
```
```python
# Compose a compact plain-English summary of the selected KPI scan results.
# Relies on variables produced earlier: df_top20_sel, matched_kpi_names_sel, matched_cmdb_count_sel, df_result_sel

try:
    top = df_top20_sel.copy()
    matched_kpis = matched_kpi_names_sel['kpi_name'].tolist()
    matched_cmdb_count = int(matched_cmdb_count_sel)
except Exception as e:
    summary = f"Error: required results not found in kernel: {e}"
else:
    # derive quick aggregates from top table
    if top.shape[0] == 0:
        summary = "No matching KPI pairs found for the given include/exclude filters or no in-window data."
    else:
        breaches_vals = sorted(top['breaches_count'].unique())
        in_window_vals = sorted(top['in_window_count'].unique())
        full_counts = sorted(top['full_series_count'].unique())
        earliest_times = [t for t in top['earliest_breach_timestamp'].unique() if t]
        sample_kpis = top['kpi_name'].unique()[:8].tolist()
        sample_cmdbs = top['cmdb_id'].unique()[:8].tolist()
        
        summary_lines = [
            "Scan summary for selected dynamic health-related KPIs (window 2022-03-21 09:00–09:30 UTC+8):",
            "",
            f"- Distinct matched KPI names (up to 50 shown): {len(matched_kpis)} (showing up to 50).",
            f"- Distinct matched cmdb_id count: {matched_cmdb_count}",
            "",
            f"- Top {min(20, top.shape[0])} (cmdb_id, kpi_name) pairs (sorted by breaches_count desc, then max_value):",
            f"  * breaches_count values in top set: {breaches_vals}",
            f"  * in_window_count values in top set: {in_window_vals}",
            f"  * full_series_count values in top set: {full_counts}",
            f"  * earliest breach times in top set (sample): {earliest_times[:3]}",
            f"  * sample KPI names in top set: {sample_kpis}",
            f"  * sample cmdb_ids in top set: {sample_cmdbs}",
            "",
            "Interpretation:",
            "- Many top pairs are container memory/cache/RSS related and show breaches for every minute in the 09:00–09:30 window",
            "  (in_window_count = breaches_count = 31, full_series_count = 61, earliest breach = 2022-03-21 09:00:00 +0800).",
            "- This pattern indicates the window values equal or exceed the global P95 (which for these pairs are large, near observed maxima),",
            "  suggesting these metrics are effectively constant/high during the window rather than transient spikes.",
            "",
            "Recommendation:",
            "- Treat timestamp-like or near-constant metrics and container start/metadata metrics as metadata (exclude or use special handling).",
            "- Focus anomaly investigation on truly dynamic metrics (CPU utilization, transient memory/working set spikes, network errors, I/O rates).",
            "- If persistent breaches on memory RSS/cache are unexpected, inspect those containers for real memory pressure or mis-labeled metrics.",
        ]
        summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                       kpi_name  full_series_count  p95_threshold  in_window_count  breaches_count  max_value_in_window  earliest_breach_timestamp
260                adservice-2          container.node-5.container_memory_rss                 61    148381696.0               31              31          148381696.0  2022-03-21 09:00:00 +0800
411               adservice2-0          container.node-6.container_memory_rss                 61    125108224.0               31              31          140013568.0  2022-03-21 09:00:00 +0800
1845            emailservice-2          container.node-6.container_memory_rss                 61     39632896.0               31              31           39632896.0  2022-03-21 09:00:00 +0800
3769   recommendationservice-2          container.node-6.container_memory_rss                 61     39612416.0               31              31           39612416.0  2022-03-21 09:00:00 +0800
3679   recommendationservice-1          container.node-6.container_memory_rss                 61     39387136.0               31              31           39387136.0  2022-03-21 09:00:00 +0800
1915           emailservice2-0          container.node-6.container_memory_rss                 61     38232064.0               31              31           38232064.0  2022-03-21 09:00:00 +0800
3859  recommendationservice2-0          container.node-6.container_memory_rss                 61     37806080.0               31              31           37806080.0  2022-03-21 09:00:00 +0800
1779            emailservice-1          container.node-6.container_memory_rss                 61     37748736.0               31              31           37748736.0  2022-03-21 09:00:00 +0800
1709            emailservice-0          container.node-6.container_memory_rss                 61     37273600.0               31              31           37273600.0  2022-03-21 09:00:00 +0800
3589   recommendationservice-0          container.node-6.container_memory_rss                 61     37154816.0               31              31           37154816.0  2022-03-21 09:00:00 +0800
3033          paymentservice-2          container.node-6.container_memory_rss                 61     34131968.0               31              31           34131968.0  2022-03-21 09:00:00 +0800
2971          paymentservice-1          container.node-6.container_memory_rss                 61     34025472.0               31              31           34025472.0  2022-03-21 09:00:00 +0800
2909          paymentservice-0          container.node-6.container_memory_rss                 61     33873920.0               31              31           33873920.0  2022-03-21 09:00:00 +0800
3099         paymentservice2-0          container.node-6.container_memory_rss                 61     33714176.0               31              31           33714176.0  2022-03-21 09:00:00 +0800
1633        currencyservice2-0          container.node-6.container_memory_rss                 61     32333824.0               31              31           33685504.0  2022-03-21 09:00:00 +0800
769             cartservice2-0        container.node-5.container_memory_cache                 61     32768000.0               31              31           32768000.0  2022-03-21 09:00:00 +0800
775             cartservice2-0  container.node-5.container_memory_mapped_file                 61     31088640.0               31              31           31088640.0  2022-03-21 09:00:00 +0800
2963          paymentservice-1        container.node-6.container_memory_cache                 61     28901376.0               31              31           28901376.0  2022-03-21 09:00:00 +0800
3851  recommendationservice2-0        container.node-6.container_memory_cache                 61     27938816.0               31              31           27938816.0  2022-03-21 09:00:00 +0800
2525               frontend2-0          container.node-6.container_memory_rss                 61     19005440.0               31              31           19005440.0  2022-03-21 09:00:00 +0800,                                              kpi_name
0   container.node-6.container_network_receive_MB....
1   container.node-6.container_network_transmit_pa...
2   container.node-6.container_network_transmit_pa...
3   container.node-6.container_network_transmit_er...
4   container.node-6.container_network_receive_err...
..                                                ...
45  container.node-6.container_fs_io_time_seconds....
46  container.node-6.container_fs_io_current./dev/...
47        container.node-6.container_cpu_user_seconds
48       container.node-6.container_cpu_usage_seconds
49   container.node-6.container_fs_usage_MB./dev/vda1

[50 rows x 1 columns], 50)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# 1) Identify trace_name values containing any target substrings
substrs = [".error_rate", ".duration_mean", ".duration_p95", ".row_count"]
pattern = '|'.join([s.replace('.', r'\.') for s in substrs])  # escape dots
mask_match = df_trace['trace_name'].str.contains(pattern, case=False, na=False)
df_trace_match = df_trace[mask_match].copy()

# 2) Compute global thresholds using ALL rows for each (cmdb_id, trace_name)
# Determine threshold type per trace_name: P95 for error_rate and duration_*, P5 for row_count
def threshold_type(name):
    name_l = name.lower()
    if '.row_count' in name_l:
        return 'P5'
    else:
        # duration_* and .error_rate -> P95
        return 'P95'

# Build thresholds DataFrame using full series of matched traces
# (Must use ALL rows for that pair - use df_trace_match which contains all matched rows)
def compute_threshold(series, thresh_type):
    if thresh_type == 'P95':
        return series.quantile(0.95)
    else:
        return series.quantile(0.05)

# Create a helper column for threshold type
df_trace_match['threshold_type'] = df_trace_match['trace_name'].apply(threshold_type)

# Compute thresholds and full_series_count per pair
df_trace_thresholds = (
    df_trace_match
    .groupby(['cmdb_id', 'trace_name', 'threshold_type'], as_index=False)['value']
    .agg(full_series_count='count', threshold_value=lambda x: compute_threshold(x, x.name if False else None))
)
# The above lambda cannot access threshold_type; compute per-group in a loop instead for correctness.

# Recompute thresholds properly (group and apply)
groups = []
for (cmdb, tname), g in df_trace_match.groupby(['cmdb_id', 'trace_name']):
    ttype = threshold_type(tname)
    if ttype == 'P95':
        thresh = g['value'].quantile(0.95)
    else:
        thresh = g['value'].quantile(0.05)
    groups.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'threshold_used': ttype,
        'threshold_value': float(thresh),
        'full_series_count': int(len(g))
    })
df_trace_thresholds = pd.DataFrame(groups)

# 3) Filter trace rows to incident window and compute in-window stats & breaches
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
mask_window = (df_trace_match['ts_sh'] >= start_sh) & (df_trace_match['ts_sh'] <= end_sh)
df_trace_window = df_trace_match[mask_window].copy()

# Merge window rows with thresholds to evaluate breaches (thresholds computed from full series)
df_window_merged = df_trace_window.merge(
    df_trace_thresholds,
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Mark breaches: for P95 use value >= threshold; for P5 use value <= threshold
def is_breach_row(row):
    if pd.isna(row['threshold_value']):
        return False
    if row['threshold_used'] == 'P95':
        return row['value'] >= row['threshold_value']
    else:
        return row['value'] <= row['threshold_value']

df_window_merged['is_breach'] = df_window_merged.apply(is_breach_row, axis=1)

# Prepare per-pair aggregations in-window
agg_in_window = df_window_merged.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    in_window_count=('value', 'count'),
    breaches_count=('is_breach', 'sum')
)

# Earliest breach timestamp and sample up to 5 unique breach timestamps per pair
df_breaches = df_window_merged[df_window_merged['is_breach']].copy()
if not df_breaches.empty:
    # earliest breach timestamp
    earliest = (
        df_breaches.groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(earliest_breach_ts=('ts_sh', 'min'))
    )
    earliest['earliest_breach_timestamp'] = earliest['earliest_breach_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    earliest = earliest.drop(columns=['earliest_breach_ts'])
    # sample unique breach timestamps (up to 5) and unique count
    sample_ts = (
        df_breaches.assign(ts_str=df_breaches['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z'))
        .groupby(['cmdb_id', 'trace_name'])['ts_str']
        .agg(lambda s: ', '.join(sorted(list(dict.fromkeys(s))[:5])))  # preserve order, take up to 5 unique
        .reset_index()
        .rename(columns={'ts_str': 'sample_breach_timestamps'})
    )
    unique_counts = (
        df_breaches.groupby(['cmdb_id', 'trace_name'])['ts_sh']
        .nunique()
        .reset_index()
        .rename(columns={'ts_sh': 'unique_breach_timestamps_count'})
    )
else:
    earliest = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'earliest_breach_timestamp'])
    sample_ts = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'sample_breach_timestamps'])
    unique_counts = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'unique_breach_timestamps_count'])

# Combine everything: thresholds + in-window aggregates + breach meta
df_pairs = (
    df_trace_thresholds
    .merge(agg_in_window, on=['cmdb_id', 'trace_name'], how='left')
    .merge(earliest, on=['cmdb_id', 'trace_name'], how='left')
    .merge(unique_counts, on=['cmdb_id', 'trace_name'], how='left')
    .merge(sample_ts, on=['cmdb_id', 'trace_name'], how='left')
)

# Fill NaNs for pairs without in-window data
df_pairs['in_window_count'] = df_pairs['in_window_count'].fillna(0).astype(int)
df_pairs['breaches_count'] = df_pairs['breaches_count'].fillna(0).astype(int)
df_pairs['unique_breach_timestamps_count'] = df_pairs['unique_breach_timestamps_count'].fillna(0).astype(int)
df_pairs['earliest_breach_timestamp'] = df_pairs['earliest_breach_timestamp'].fillna('')
df_pairs['sample_breach_timestamps'] = df_pairs['sample_breach_timestamps'].fillna('')

# 4a) Prepare output table sorted by unique_breach_timestamps_count desc then breaches_count desc, limit 50 rows
out_table = df_pairs.sort_values(['unique_breach_timestamps_count', 'breaches_count'], ascending=[False, False])[
    [
        'cmdb_id', 'trace_name', 'full_series_count', 'threshold_used', 'threshold_value',
        'in_window_count', 'breaches_count', 'earliest_breach_timestamp',
        'unique_breach_timestamps_count', 'sample_breach_timestamps'
    ]
].head(50)

# 4b) Aggregate tally of breach frequency by timestamp within the window:
# For each timestamp in window, count distinct (cmdb_id, trace_name) pairs that had a breach at that timestamp
if not df_breaches.empty:
    df_breaches['ts_str'] = df_breaches['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    breach_tally = (
        df_breaches.groupby('ts_str')
        .apply(lambda g: g[['cmdb_id', 'trace_name']].drop_duplicates().shape[0])
        .reset_index(name='number_of_pairs_with_breach_at_that_timestamp')
        .sort_values('number_of_pairs_with_breach_at_that_timestamp', ascending=False)
        .head(20)
    )
else:
    breach_tally = pd.DataFrame(columns=['ts_str', 'number_of_pairs_with_breach_at_that_timestamp'])

# Keep intermediate variables in kernel memory:
# df_trace_match, df_trace_thresholds, df_trace_window, df_window_merged, df_breaches, df_pairs

out_table, breach_tally

```
Out[5]:
```
```python
# Compose a concise plain-English summary of the trace analysis using existing kernel results (out_table, breach_tally, df_pairs).
import pandas as pd

try:
    out_table
    breach_tally
    df_pairs
except NameError as e:
    summary = f"Error: required analysis tables not found in kernel: {e}"
else:
    total_pairs = int(df_pairs.shape[0])
    pairs_with_any_breach = int((df_pairs['breaches_count'] > 0).sum())
    top_pairs_shown = min(50, out_table.shape[0])
    
    # Top timestamps by number of pairs with breach (take top 5 for summary)
    top_ts = breach_tally.head(5).to_dict(orient='records')
    top_ts_lines = [f"{r['ts_str']}: {int(r['number_of_pairs_with_breach_at_that_timestamp'])} pairs" for r in top_ts]
    
    # Threshold usage counts
    thresh_counts = df_pairs['threshold_used'].value_counts().to_dict()
    
    # Sample trace_name patterns from top table
    sample_trace_names = out_table['trace_name'].unique()[:6].tolist()
    
    summary_lines = [
        "Trace analysis summary (window 2022-03-21 09:00:00 to 09:30:00, Asia/Shanghai / UTC+8):",
        "",
        f"- Total matched (cmdb_id, trace_name) pairs examined: {total_pairs}",
        f"- Pairs with >=1 breach in-window: {pairs_with_any_breach}",
        f"- Top table returned: showing up to {top_pairs_shown} pairs sorted by unique breach timestamps then breaches.",
        "",
        "Key findings:",
        "- A very large number of trace pairs show breaches of their global thresholds during the entire 09:00–09:30 window.",
        "- Most breaches are for '.error_rate' and use P95 thresholds (thresholds reported as P95); a smaller set would use P5 for '.row_count'.",
        f"- Threshold type counts in results: {thresh_counts}",
        "",
        "Top timestamps with most simultaneous breaches (top 5):",
    ] + [f"  * {line}" for line in top_ts_lines] + [
        "",
        f"- Sample trace_name patterns among top pairs: {sample_trace_names}",
        "",
        "Interpretation:",
        "- Many trace.error_rate series have global P95 thresholds equal to 0.0, causing every nonzero/any value in the window to be counted as a breach.",
        "- Numerous pairs show breaches every minute in the window (in_window_count = breaches_count = 30 for many rows),",
        "  indicating systemic failures across many service-to-service calls rather than isolated spikes.",
        "- The peak coordinated breach counts occur around 09:10–09:12 (highest at 2022-03-21 09:11:00 +0800 with the most pairs affected).",
        "",
        "Recommendation:",
        "- Investigate cross-service connectivity or a common dependency (frontend/back-end calls) active around 09:10–09:12.",
        "- Review whether P95 thresholds of 0.0 for error_rate are meaningful (historical zeros can make P95=0); consider using nonzero baselines or alternative rules for sparse error metrics.",
        "- Prioritize tracing and logs for calls listed in the top table (many are from_frontend.* and to_* service traces) to identify a root cause.",
        ""
    ]
    
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                   trace_name  full_series_count threshold_used  threshold_value  in_window_count  breaches_count  earliest_breach_timestamp  unique_breach_timestamps_count                           sample_breach_timestamps
2    adservice-0             trace.from_frontend-0.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
6    adservice-0             trace.from_frontend-1.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
10   adservice-0             trace.from_frontend-2.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
14   adservice-1             trace.from_frontend-0.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
18   adservice-1             trace.from_frontend-1.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
..           ...                                          ...                ...            ...              ...              ...             ...                        ...                             ...                                                ...
594   frontend-0  trace.to_recommendationservice-1.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
598   frontend-0  trace.to_recommendationservice-2.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
602   frontend-0        trace.to_shippingservice-0.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
606   frontend-0        trace.to_shippingservice-1.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...
610   frontend-0        trace.to_shippingservice-2.error_rate                 30            P95              0.0               30              30  2022-03-21 09:00:00 +0800                              30  2022-03-21 09:00:00 +0800, 2022-03-21 09:01:00...

[50 rows x 10 columns],                        ts_str  number_of_pairs_with_breach_at_that_timestamp
11  2022-03-21 09:11:00 +0800                                            644
10  2022-03-21 09:10:00 +0800                                            469
8   2022-03-21 09:08:00 +0800                                            441
9   2022-03-21 09:09:00 +0800                                            425
15  2022-03-21 09:15:00 +0800                                            380
7   2022-03-21 09:07:00 +0800                                            378
28  2022-03-21 09:28:00 +0800                                            374
27  2022-03-21 09:27:00 +0800                                            374
6   2022-03-21 09:06:00 +0800                                            372
1   2022-03-21 09:01:00 +0800                                            372
22  2022-03-21 09:22:00 +0800                                            370
20  2022-03-21 09:20:00 +0800                                            364
23  2022-03-21 09:23:00 +0800                                            360
24  2022-03-21 09:24:00 +0800                                            360
3   2022-03-21 09:03:00 +0800                                            354
4   2022-03-21 09:04:00 +0800                                            354
17  2022-03-21 09:17:00 +0800                                            350
0   2022-03-21 09:00:00 +0800                                            346
29  2022-03-21 09:29:00 +0800                                            342
26  2022-03-21 09:26:00 +0800                                            342)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing dataframes if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime columns (Asia/Shanghai / UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(tz)

# Step 1: Trace breach tally per timestamp for specified patterns
trace_subs = [r"\.error_rate", r"\.duration_mean", r"\.duration_p95", r"\.row_count"]
trace_pattern = "|".join(trace_subs)
df_trace_match = df_trace[df_trace['trace_name'].str.contains(trace_pattern, case=False, na=False)].copy()

# Compute global thresholds per (cmdb_id, trace_name) using ALL rows
def thresh_type(name):
    nl = name.lower()
    return 'P5' if '.row_count' in nl else 'P95'

groups = []
for (cmdb, tname), g in df_trace_match.groupby(['cmdb_id', 'trace_name']):
    ttype = thresh_type(tname)
    if ttype == 'P95':
        thresh = float(g['value'].quantile(0.95))
    else:
        thresh = float(g['value'].quantile(0.05))
    groups.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'threshold_used': ttype,
        'threshold_value': thresh,
        'full_series_count': int(len(g))
    })
df_trace_thresholds = pd.DataFrame(groups)

# Filter to incident window
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
mask_window = (df_trace_match['ts_sh'] >= start_sh) & (df_trace_match['ts_sh'] <= end_sh)
df_trace_window = df_trace_match[mask_window].copy()

# Merge thresholds and evaluate breaches per row
df_tw = df_trace_window.merge(df_trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

def is_breach(r):
    if pd.isna(r['threshold_value']):
        return False
    if r['threshold_used'] == 'P95':
        return r['value'] >= r['threshold_value']
    else:
        return r['value'] <= r['threshold_value']

df_tw['is_breach'] = df_tw.apply(is_breach, axis=1)

# Count distinct (cmdb_id, trace_name) per timestamp that had a breach
df_tw['ts_str'] = df_tw['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
df_breach_pairs = df_tw[df_tw['is_breach']][['ts_str', 'cmdb_id', 'trace_name']].drop_duplicates()
breach_tally = (
    df_breach_pairs.groupby('ts_str')
    .size()
    .reset_index(name='number_of_pairs_with_breach_at_that_timestamp')
    .sort_values('number_of_pairs_with_breach_at_that_timestamp', ascending=False)
)
top3_breach_timestamps_df = breach_tally.head(3).reset_index(drop=True)

# Step 2: Metric dynamic KPI selection and global P95 per (cmdb_id,kpi_name)
includes = [
    "cpu", "usage", "user_seconds", "system_seconds", "mem", "memory", "rss",
    "working_set", "heap", "gc", "network", "transmit", "receive", "retransmit",
    "packet", "err", "drop", "latency", "delay", "fs_reads", "fs_writes",
    "io", "read", "write", "disk", "io_time", "tasks", "process"
]
excludes = [
    "start_time", "starttime", "runtime.java_lang", "garbagecollector",
    "memorypool", "system.disk.total", "committedvirtual", "totalphysical"
]
inc_pat = "|".join(re.escape(s) for s in includes)
exc_pat = "|".join(re.escape(s) for s in excludes)
mask_inc = df_metric['kpi_name'].str.contains(inc_pat, case=False, na=False)
mask_exc = df_metric['kpi_name'].str.contains(exc_pat, case=False, na=False)
mask_sel = mask_inc & (~mask_exc)
df_metric_sel = df_metric[mask_sel].copy()

# Compute global P95 per (cmdb_id,kpi_name) using ALL rows
df_metric_thresholds = (
    df_metric_sel
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(full_series_count='count', p95_threshold=lambda x: float(x.quantile(0.95)))
)

# Step 3: For each top-3 breach timestamp, extract metric values at that timestamp and compute deviation_ratio
# Prepare list of timestamps (as timezone-aware)
top_timestamps = top3_breach_timestamps_df['ts_str'].tolist()

rows_list = []
distinct_counts = []
for ts_str in top_timestamps:
    # parse back to timestamp
    ts = pd.to_datetime(ts_str).tz_convert(tz)  # already includes offset
    # select metric rows at this exact timestamp
    df_at_ts = df_metric_sel[df_metric_sel['ts_sh'] == ts].copy()
    if df_at_ts.empty:
        # no metric rows at this timestamp
        continue
    # merge with thresholds (global P95)
    df_at_ts = df_at_ts.merge(df_metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    # compute deviation_ratio; handle p95 == 0
    df_at_ts['p95_is_zero'] = df_at_ts['p95_threshold'].fillna(0) == 0
    # when p95 is zero -> set ratio to +inf per requirement
    def compute_ratio(row):
        p95 = row['p95_threshold']
        val = row['value']
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf
        return float(val) / float(p95)
    df_at_ts['deviation_ratio'] = df_at_ts.apply(compute_ratio, axis=1)
    # Mark p95==0 flag
    df_at_ts['p95_eq_zero'] = df_at_ts['p95_threshold'] == 0
    # Select top 10 by deviation_ratio desc (inf will be on top)
    df_top10 = df_at_ts.sort_values('deviation_ratio', ascending=False).head(10)
    # Prepare output rows with required columns and add timestamp column
    df_top10_out = df_top10[[
        'cmdb_id', 'kpi_name', 'value', 'p95_threshold', 'deviation_ratio', 'p95_eq_zero'
    ]].copy()
    df_top10_out.insert(0, 'analysis_timestamp', ts_str)
    rows_list.append(df_top10_out)
    distinct_counts.append({
        'analysis_timestamp': ts_str,
        'distinct_cmdb_id_count_among_top10': int(df_top10_out['cmdb_id'].nunique())
    })

# Combine per-timestamp top10 results into one DataFrame (if any)
if rows_list:
    metrics_top_by_timestamp_df = pd.concat(rows_list, ignore_index=True)
else:
    metrics_top_by_timestamp_df = pd.DataFrame(columns=[
        'analysis_timestamp','cmdb_id','kpi_name','value','p95_threshold','deviation_ratio','p95_eq_zero'
    ])

distinct_cmdb_counts_df = pd.DataFrame(distinct_counts)

# Keep intermediate variables in kernel:
# df_trace_match, df_trace_thresholds, df_trace_window, df_tw, df_breach_pairs, breach_tally,
# df_metric_sel, df_metric_thresholds, metrics_top_by_timestamp_df, distinct_cmdb_counts_df

# Final compact outputs
top3_breach_timestamps_df, metrics_top_by_timestamp_df, distinct_cmdb_counts_df

```
Out[6]:
```
# Compose a compact plain-English summary from the computed results (assumes variables from previous cells exist).
summary = (
    "Summary of analysis:\n\n"
    "1) Top 3 timestamps with the most trace-pair breaches (UTC+8):\n"
    "   - 2022-03-21 09:11:00 +0800: 644 pairs\n"
    "   - 2022-03-21 09:10:00 +0800: 469 pairs\n"
    "   - 2022-03-21 09:08:00 +0800: 441 pairs\n\n"
    "2) Metric deviation check at each of these timestamps (top-10 per timestamp):\n"
    "   - Many of the top metric entries at these timestamps have global P95 == 0, which yields deviation_ratio = +Inf.\n"
    "     These infinite ratios indicate the selected P95 baseline is zero (sparse/zero historical values), so breaches may be\n"
    "     an artifact of thresholding rather than large absolute spikes.\n"
    "   - The top deviating metric pairs (by ratio) concentrate on a small set of cmdb_ids (distinct cmdb_id count among top-10 = 2\n"
    "     for each timestamp), notably shippingservice2-0 and adservice in the top results.\n"
    "   - Example metric types appearing in the top entries include network error/packet counters, container memory failure or RSS/cache\n"
    "     metrics and some runtime/buffer metrics — these should be inspected for correctness and semantics.\n\n"
    "3) Interpretation & recommended next steps:\n"
    "   - There is a coordinated spike of trace breaches centered around 09:08–09:11, peaking at 09:11. Investigate service interactions\n"
    "     around that interval (trace error spikes previously observed) as a likely temporal failure signature.\n"
    "   - For metrics with p95==0, treat breaches as special cases (use alternative baselines or require sustained nonzero behavior before\n"
    "     flagging). Focus on metrics with meaningful nonzero P95 for true resource anomalies.\n"
    "   - Prioritize investigation of shippingservice2 and adservice (and their network/error-related and memory-related metrics) for\n"
    "     root-cause signals around 09:10–09:11, and correlate with traces and logs from the same interval.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      ts_str  number_of_pairs_with_breach_at_that_timestamp
0  2022-03-21 09:11:00 +0800                                            644
1  2022-03-21 09:10:00 +0800                                            469
2  2022-03-21 09:08:00 +0800                                            441,            analysis_timestamp             cmdb_id                                           kpi_name  value  p95_threshold  deviation_ratio  p95_eq_zero
0   2022-03-21 09:11:00 +0800  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    0.0            0.0              inf         True
1   2022-03-21 09:11:00 +0800  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    0.0            0.0              inf         True
2   2022-03-21 09:11:00 +0800           adservice           runtime.java_nio_BufferPool_Count.mapped    0.0            0.0              inf         True
3   2022-03-21 09:11:00 +0800  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    0.0            0.0              inf         True
4   2022-03-21 09:11:00 +0800           adservice      runtime.java_nio_BufferPool_MemoryUsed.mapped    0.0            0.0              inf         True
..                        ...                 ...                                                ...    ...            ...              ...          ...
25  2022-03-21 09:08:00 +0800  shippingservice2-0  container.node-6.container_network_receive_err...    0.0            0.0              inf         True
26  2022-03-21 09:08:00 +0800  shippingservice2-0  container.node-6.container_network_receive_pac...    0.0            0.0              inf         True
27  2022-03-21 09:08:00 +0800  shippingservice2-0  container.node-6.container_network_transmit_er...    0.0            0.0              inf         True
28  2022-03-21 09:08:00 +0800  shippingservice2-0  container.node-6.container_memory_failures.con...    0.0            0.0              inf         True
29  2022-03-21 09:08:00 +0800  shippingservice2-0  container.node-6.container_memory_failures.con...    0.0            0.0              inf         True

[30 rows x 7 columns],           analysis_timestamp  distinct_cmdb_id_count_among_top10
0  2022-03-21 09:11:00 +0800                                   2
1  2022-03-21 09:10:00 +0800                                   2
2  2022-03-21 09:08:00 +0800                                   2)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re

# Reuse existing df_trace and df_trace_thresholds if present; otherwise load/recompute
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# ensure df_trace_thresholds exists (computed previously); if not, compute thresholds for .error_rate traces
if 'df_trace_thresholds' not in globals():
    # compute thresholds for matching trace_name patterns (use all rows)
    mask_err = df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)
    df_trace_err = df_trace[mask_err].copy()
    groups = []
    for (cmdb, tname), g in df_trace_err.groupby(['cmdb_id', 'trace_name']):
        thresh = float(g['value'].quantile(0.95))
        groups.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'threshold_used': 'P95',
            'threshold_value': thresh,
            'full_series_count': int(len(g))
        })
    df_trace_thresholds = pd.DataFrame(groups)

# Target timestamps (Asia/Shanghai tz-aware)
ts_list = [
    tz.localize(pd.Timestamp('2022-03-21 09:11:00')),
    tz.localize(pd.Timestamp('2022-03-21 09:27:00'))
]
ts_strs = [t.strftime('%Y-%m-%d %H:%M:%S %z') for t in ts_list]

# Helper to extract service name from trace_name
def extract_service(trace_name):
    tn = str(trace_name)
    m = re.search(r'\bto_([a-z0-9\-]+)', tn, flags=re.I)
    if m:
        svc = m.group(1)
    else:
        m = re.search(r'\bfrom_([a-z0-9\-]+)', tn, flags=re.I)
        if m:
            svc = m.group(1)
        else:
            m = re.search(r'trace\.([a-z0-9\-]+)', tn, flags=re.I)
            if m:
                svc = m.group(1)
            else:
                # fallback: try any token like to<service> or from<service>
                m = re.search(r'\b([a-z0-9]+service)[\-\.]?', tn, flags=re.I)
                svc = m.group(1) if m else 'unknown'
    return svc.split('-')[0].lower()

# Containers for results
agg_results = {}
total_breached_counts = {}

for ts, ts_str in zip(ts_list, ts_strs):
    # Filter trace rows at timestamp and with .error_rate
    df_at_ts = df_trace[(df_trace['ts_sh'] == ts) & df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)].copy()
    if df_at_ts.empty:
        agg_results[ts_str] = pd.DataFrame(columns=[
            'service_name', 'total_breached_trace_pairs_at_timestamp',
            'distinct_cmdb_ids_involved', 'sample_trace_names'
        ])
        total_breached_counts[ts_str] = 0
        continue

    # Merge with global thresholds
    df_at_ts = df_at_ts.merge(df_trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

    # Determine breach: if threshold_value==0 -> breach if value>0; else value >= threshold_value
    def is_breach_row(r):
        thr = r['threshold_value']
        val = r['value']
        if pd.isna(thr):
            return False
        if thr == 0:
            return val > 0
        return val >= thr

    df_at_ts['is_breach'] = df_at_ts.apply(is_breach_row, axis=1)

    # Keep only breached rows
    df_breached = df_at_ts[df_at_ts['is_breach']].copy()

    # Extract service name
    df_breached['service_name'] = df_breached['trace_name'].apply(extract_service)

    # For counting unique trace pairs, ensure uniqueness by (cmdb_id, trace_name)
    df_breached_unique = df_breached[['service_name','cmdb_id','trace_name']].drop_duplicates()

    # Aggregate by service_name
    agg = (
        df_breached_unique
        .groupby('service_name', as_index=False)
        .agg(
            total_breached_trace_pairs_at_timestamp = ('trace_name', 'count'),
            distinct_cmdb_ids = ('cmdb_id', lambda s: ', '.join(sorted(list(dict.fromkeys(s))[:10]))),
            sample_trace_names = ('trace_name', lambda s: ', '.join(list(dict.fromkeys(s))[:10]))
        )
    )

    # Sort by breached pairs desc and take top 10
    agg = agg.sort_values('total_breached_trace_pairs_at_timestamp', ascending=False).head(10).reset_index(drop=True)

    # Rename columns per requirement and store
    agg = agg.rename(columns={'distinct_cmdb_ids': 'distinct_cmdb_ids_involved'})
    agg_results[ts_str] = agg[['service_name','total_breached_trace_pairs_at_timestamp','distinct_cmdb_ids_involved','sample_trace_names']]

    # total breached pairs at timestamp (unique pairs)
    total_breached_counts[ts_str] = int(df_breached_unique.shape[0])

# Expose results (compact)
agg_results['2022-03-21 09:11:00 +0800'.replace('', '')], agg_results['2022-03-21 09:27:00 +0800'.replace('', '')], total_breached_counts

```
Out[7]:
```
summary = (
    "Result summary (plain English):\n\n"
    "- For the two timestamps checked (2022-03-21 09:11:00 +0800 and 2022-03-21 09:27:00 +0800),\n"
    "  after filtering trace.csv to rows at each timestamp with '.error_rate' and applying the full-series P95\n"
    "  breach rule (and treating P95==0 as breach only if value>0), there were NO breached trace pairs found.\n\n"
    "- Service-level aggregation is therefore empty for both timestamps, and the total number of breached trace pairs\n"
    "  at each timestamp is 0.\n\n"
    "Short checks/recommendations if this is unexpected:\n"
    "1) Confirm the timestamp/timezone alignment and that trace rows exist at those exact minutes.\n"
    "2) Verify the global P95 thresholds used (they must be the same threshold table computed earlier).\n"
    "3) If prior analyses showed many breaches at nearby times, re-run with a small time window around these minutes\n"
    "   or check for differences in trace_name matching patterns.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [service_name, total_breached_trace_pairs_at_timestamp, distinct_cmdb_ids_involved, sample_trace_names]
Index: [], Empty DataFrame
Columns: [service_name, total_breached_trace_pairs_at_timestamp, distinct_cmdb_ids_involved, sample_trace_names]
Index: [], {'2022-03-21 09:11:00 +0800': 0, '2022-03-21 09:27:00 +0800': 0})```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware datetime column in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# 1) Compute global P95 thresholds for all (cmdb_id, trace_name) pairs of interest ('.error_rate') using ALL rows
mask_error_rate_all = df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)
df_trace_error_all = df_trace[mask_error_rate_all].copy()

# If there are no error_rate rows, return empty results
if df_trace_error_all.empty:
    top20_aggs = pd.DataFrame(columns=['timestamp','total_rows','count_value_gt_zero','breaches_count'])
    sample_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_threshold'])
else:
    df_thresholds = (
        df_trace_error_all
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .agg(full_series_count='count', p95_threshold=lambda x: float(x.quantile(0.95)))
    )

    # 2) Filter trace rows to window 2022-03-21 09:00:00 to 2022-03-21 09:30:00 (inclusive) and containing '.error_rate'
    start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
    end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
    mask_window = (df_trace['ts_sh'] >= start_sh) & (df_trace['ts_sh'] <= end_sh) & df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)
    df_window = df_trace[mask_window].copy()

    # Merge thresholds (global) into window rows
    df_window = df_window.merge(df_thresholds, on=['cmdb_id', 'trace_name'], how='left')

    # 3) Determine breaches per-row using rule: if p95 is NaN -> not breach; if p95 == 0 -> breach if value > 0; else breach if value >= p95
    def row_is_breach(r):
        thr = r['p95_threshold']
        val = r['value']
        if pd.isna(thr):
            return False
        if thr == 0:
            return val > 0
        return val >= thr

    df_window['is_breach'] = df_window.apply(row_is_breach, axis=1)

    # Prepare per-minute aggregates
    # total_rows: count of matching rows at that minute
    total_rows = df_window.groupby('ts_sh').size().rename('total_rows').reset_index()
    # count_value_gt_zero: count of rows with value > 0
    count_val_gt0 = df_window[df_window['value'] > 0].groupby('ts_sh').size().rename('count_value_gt_zero').reset_index()
    # breaches_count: number of unique (cmdb_id, trace_name) pairs that are breached at that minute
    df_breached_pairs = df_window[df_window['is_breach']][['ts_sh','cmdb_id','trace_name']].drop_duplicates()
    breaches_count = df_breached_pairs.groupby('ts_sh').size().rename('breaches_count').reset_index()

    # Merge aggregates into one table
    agg = total_rows.merge(count_val_gt0, on='ts_sh', how='left').merge(breaches_count, on='ts_sh', how='left')
    agg['count_value_gt_zero'] = agg['count_value_gt_zero'].fillna(0).astype(int)
    agg['breaches_count'] = agg['breaches_count'].fillna(0).astype(int)

    # Format timestamp as string with timezone for output
    agg['timestamp'] = agg['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    top20_aggs = agg.sort_values('breaches_count', ascending=False)[['timestamp','total_rows','count_value_gt_zero','breaches_count']].head(20).reset_index(drop=True)

    # 4) Sample up to 20 rows from the single minute that has the highest breaches_count
    if top20_aggs.shape[0] == 0:
        sample_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_threshold'])
    else:
        top_ts_str = top20_aggs.at[0, 'timestamp']
        # convert back to tz-aware timestamp for matching
        top_ts = pd.to_datetime(top_ts_str).tz_convert(tz)
        df_top_minute = df_window[df_window['ts_sh'] == top_ts].copy()
        # select sample columns and include p95_threshold
        sample_rows = df_top_minute[['ts_sh','cmdb_id','trace_name','value','p95_threshold']].copy()
        sample_rows['timestamp'] = sample_rows['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
        sample_rows = sample_rows[['timestamp','cmdb_id','trace_name','value','p95_threshold']].head(20).reset_index(drop=True)

# Final compact outputs
top20_aggs, sample_rows

```
Out[8]:
```
```python
# Compose a concise plain-English summary from the last computed results (top20_aggs, sample_rows).
try:
    top = top20_aggs
    sample = sample_rows
except NameError:
    summary = "Error: required result tables (top20_aggs, sample_rows) not found in kernel."
else:
    if top.shape[0] == 0:
        summary = "No .error_rate rows found in the specified window (2022-03-21 09:00–09:30 +0800)."
    else:
        # Top minute
        top_min = top.iloc[0]
        top_ts = top_min['timestamp']
        top_total = int(top_min['total_rows'])
        top_gt0 = int(top_min['count_value_gt_zero'])
        top_breaches = int(top_min['breaches_count'])

        # Count of minutes with any breaches in the top-20 list
        minutes_with_breaches = int((top['breaches_count'] > 0).sum())

        summary_lines = [
            "Summary (trace .error_rate per-minute aggregates, window 2022-03-21 09:00–09:30 +0800):",
            "",
            f"- Top minute by breaches: {top_ts}",
            f"  * total matching rows at that minute: {top_total}",
            f"  * rows with value > 0 at that minute: {top_gt0}",
            f"  * breached trace pairs at that minute: {top_breaches}",
            "",
            f"- Across the examined minutes (top 20 minutes shown), minutes with any breaches: {minutes_with_breaches}.",
            "- Most minutes in the window show zero breaches because most trace.error_rate values are 0 and global P95 thresholds are 0;",
            "  our rule treats P95==0 as a breach only when value>0, so zeros do not trigger breaches.",
            "",
            "- Actionable next step: inspect the nonzero / breached trace pair(s) occurring at the top minute (09:29:00 +0800) using the sample rows",
            "  returned for that minute (these sample rows are available as `sample_rows`) to identify the specific (cmdb_id, trace_name) responsible.",
            ""
        ]
        summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp  total_rows  count_value_gt_zero  breaches_count
0   2022-03-21 09:29:00 +0800         204                    1               1
1   2022-03-21 09:00:00 +0800         238                    0               0
2   2022-03-21 09:02:00 +0800         226                    0               0
3   2022-03-21 09:03:00 +0800         234                    0               0
4   2022-03-21 09:04:00 +0800         238                    0               0
5   2022-03-21 09:01:00 +0800         236                    0               0
6   2022-03-21 09:06:00 +0800         246                    0               0
7   2022-03-21 09:07:00 +0800         216                    0               0
8   2022-03-21 09:08:00 +0800         240                    0               0
9   2022-03-21 09:09:00 +0800         228                    0               0
10  2022-03-21 09:10:00 +0800         184                    0               0
11  2022-03-21 09:11:00 +0800         240                    0               0
12  2022-03-21 09:12:00 +0800         202                    0               0
13  2022-03-21 09:05:00 +0800         216                    0               0
14  2022-03-21 09:13:00 +0800         244                    0               0
15  2022-03-21 09:14:00 +0800         228                    0               0
16  2022-03-21 09:16:00 +0800         216                    0               0
17  2022-03-21 09:15:00 +0800         256                    0               0
18  2022-03-21 09:18:00 +0800         232                    0               0
19  2022-03-21 09:19:00 +0800         228                    0               0,                     timestamp        cmdb_id                               trace_name  value  p95_threshold
0   2022-03-21 09:29:00 +0800    adservice-0         trace.from_frontend-0.error_rate    0.0            0.0
1   2022-03-21 09:29:00 +0800    adservice-0         trace.from_frontend-1.error_rate    0.0            0.0
2   2022-03-21 09:29:00 +0800    adservice-0         trace.from_frontend-2.error_rate    0.0            0.0
3   2022-03-21 09:29:00 +0800    adservice-1         trace.from_frontend-0.error_rate    0.0            0.0
4   2022-03-21 09:29:00 +0800    adservice-1         trace.from_frontend-1.error_rate    0.0            0.0
5   2022-03-21 09:29:00 +0800    adservice-1         trace.from_frontend-2.error_rate    0.0            0.0
6   2022-03-21 09:29:00 +0800    adservice-2         trace.from_frontend-0.error_rate    0.0            0.0
7   2022-03-21 09:29:00 +0800    adservice-2         trace.from_frontend-1.error_rate    0.0            0.0
8   2022-03-21 09:29:00 +0800    adservice-2         trace.from_frontend-2.error_rate    0.0            0.0
9   2022-03-21 09:29:00 +0800   adservice2-0        trace.from_frontend2-0.error_rate    0.0            0.0
10  2022-03-21 09:29:00 +0800  cartservice-0  trace.from_checkoutservice-1.error_rate    0.0            0.0
11  2022-03-21 09:29:00 +0800  cartservice-0  trace.from_checkoutservice-2.error_rate    0.0            0.0
12  2022-03-21 09:29:00 +0800  cartservice-0         trace.from_frontend-0.error_rate    0.0            0.0
13  2022-03-21 09:29:00 +0800  cartservice-0         trace.from_frontend-1.error_rate    0.0            0.0
14  2022-03-21 09:29:00 +0800  cartservice-0         trace.from_frontend-2.error_rate    0.0            0.0
15  2022-03-21 09:29:00 +0800  cartservice-0                    trace.self.error_rate    0.0            0.0
16  2022-03-21 09:29:00 +0800  cartservice-1  trace.from_checkoutservice-1.error_rate    0.0            0.0
17  2022-03-21 09:29:00 +0800  cartservice-1         trace.from_frontend-0.error_rate    0.0            0.0
18  2022-03-21 09:29:00 +0800  cartservice-1         trace.from_frontend-1.error_rate    0.0            0.0
19  2022-03-21 09:29:00 +0800  cartservice-1         trace.from_frontend-2.error_rate    0.0            0.0)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# 1) Compute full-series P95 thresholds for all (cmdb_id, trace_name) pairs with ".error_rate" using ALL rows (global thresholds)
mask_error_rate_all = df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)
df_error_all = df_trace[mask_error_rate_all].copy()

if df_error_all.empty:
    # No error_rate series at all -> return empty result with required columns
    result = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_threshold'])
else:
    df_thresholds_err = (
        df_error_all
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(p95_threshold=lambda x: float(x.quantile(0.95)))
    )

    # 2) Filter to the exact minute 2022-03-21 09:29:00 (Asia/Shanghai) and trace_name contains ".error_rate" and value > 0
    target_ts = tz.localize(pd.Timestamp('2022-03-21 09:29:00'))
    mask_minute = (df_trace['ts_sh'] == target_ts) & df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False) & (df_trace['value'] > 0)
    df_at_minute = df_trace[mask_minute].copy()

    if df_at_minute.empty:
        result = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_threshold'])
    else:
        # 3) Merge with global P95 thresholds
        df_merged = df_at_minute.merge(df_thresholds_err, on=['cmdb_id','trace_name'], how='left')

        # Format timestamp column as string with timezone for compact output
        df_merged['timestamp'] = df_merged['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')

        # Select required columns and limit to 50 rows
        result = df_merged[['timestamp','cmdb_id','trace_name','value','p95_threshold']].head(50).reset_index(drop=True)

# Display the compact result DataFrame (variable name used for output)
result

```
Out[9]:
```
summary = (
    "At 2022-03-21 09:29:00 +0800 there is one trace.error_rate row with value>0:\n"
    "- cmdb_id: frontend2-0\n"
    "- trace_name: trace.self.error_rate\n"
    "- value: 0.007874\n"
    "- full-series P95 threshold: 0.007874\n\n"
    "Because the value equals the P95 (>=), this is counted as a breach for that pair at that minute. "
    "No other .error_rate rows with value>0 were found at that timestamp. "
    "Suggested next step: inspect traces and logs for frontend2-0 around 09:29 to understand the error source."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp      cmdb_id             trace_name     value  p95_threshold
0  2022-03-21 09:29:00 +0800  frontend2-0  trace.self.error_rate  0.007874       0.007874```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# reuse df_trace if exists, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# Filter conditions
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))

mask = (
    df_trace['trace_name'].str.contains(r'\.error_rate', case=False, na=False)
) & (
    df_trace['ts_sh'] >= start_sh
) & (
    df_trace['ts_sh'] <= end_sh
) & (
    df_trace['value'] > 0
)

df_filtered = df_trace[mask].copy()

# Prepare compact sample table (sorted ascending by timestamp) and format timestamp strings
if not df_filtered.empty:
    df_filtered = df_filtered.sort_values('ts_sh', ascending=True)
    df_filtered['timestamp'] = df_filtered['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
    sample_rows = df_filtered[['timestamp', 'cmdb_id', 'trace_name', 'value']].head(50).reset_index(drop=True)
    distinct_timestamps_count = int(df_filtered['ts_sh'].nunique())
    earliest_ts = df_filtered['ts_sh'].min().strftime('%Y-%m-%d %H:%M:%S %z')
else:
    sample_rows = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'trace_name', 'value'])
    distinct_timestamps_count = 0
    earliest_ts = ''

# Return compact outputs: sample rows (<=50), distinct timestamp count, earliest timestamp string (or empty)
sample_rows, distinct_timestamps_count, earliest_ts

```
Out[10]:
```
Summary:

- Matching rows (trace_name contains ".error_rate", value > 0) within 2022-03-21 09:00–09:30 +08:
  - 2022-03-21 09:29:00 +0800 | cmdb_id: frontend2-0 | trace_name: trace.self.error_rate | value: 0.007874

- Aggregates:
  1) Number of distinct timestamps with value>0 in the window: 1
  2) Earliest timestamp with value>0: 2022-03-21 09:29:00 +0800

No other .error_rate rows with value>0 were found in that window.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp      cmdb_id             trace_name     value
0  2022-03-21 09:29:00 +0800  frontend2-0  trace.self.error_rate  0.007874, 1, '2022-03-21 09:29:00 +0800')```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse existing df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# Patterns for duration metrics
pattern_duration = r'\.duration_mean|\.duration_p95'

# 1) Identify all duration rows across the full series (for threshold calculation)
mask_duration_all = df_trace['trace_name'].str.contains(pattern_duration, case=False, na=False)
df_duration_all = df_trace[mask_duration_all].copy()

if df_duration_all.empty:
    # Prepare empty outputs with required columns if no matches
    top10_pairs = pd.DataFrame(columns=[
        'cmdb_id','trace_name','full_series_count','p95_threshold',
        'in_window_count','breaches_count','earliest_breach_timestamp','max_value_in_window'
    ])
    breach_tally_top10 = pd.DataFrame(columns=['timestamp','number_of_pairs_with_breach_at_that_timestamp'])
else:
    # 2) Compute global P95 for each (cmdb_id, trace_name) using ALL available rows
    df_trace_thresholds_duration = (
        df_duration_all
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .agg(full_series_count='count', p95_threshold=lambda x: float(x.quantile(0.95)))
    )

    # 3) Filter to incident window (inclusive) and duration traces
    start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
    end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
    mask_window = (df_trace['ts_sh'] >= start_sh) & (df_trace['ts_sh'] <= end_sh) & mask_duration_all
    df_window_duration = df_trace[mask_window].copy()

    # Merge thresholds into window rows
    df_window_merged_duration = df_window_duration.merge(
        df_trace_thresholds_duration,
        on=['cmdb_id','trace_name'],
        how='left'
    )

    # Determine breaches: value >= p95_threshold (NaN threshold -> not breach)
    df_window_merged_duration['is_breach'] = df_window_merged_duration['value'] >= df_window_merged_duration['p95_threshold']
    df_window_merged_duration['is_breach'] = df_window_merged_duration['is_breach'].fillna(False)

    # Aggregations per (cmdb_id, trace_name) in-window
    agg_window = df_window_merged_duration.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        in_window_count=('value','count'),
        breaches_count=('is_breach','sum'),
        max_value_in_window=('value','max')
    )

    # Earliest breach timestamp per pair (if any)
    df_breaches_duration = df_window_merged_duration[df_window_merged_duration['is_breach']].copy()
    if not df_breaches_duration.empty:
        earliest = (
            df_breaches_duration.groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(earliest_breach_ts=('ts_sh','min'))
        )
        earliest['earliest_breach_timestamp'] = earliest['earliest_breach_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
        earliest = earliest.drop(columns=['earliest_breach_ts'])
    else:
        earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_breach_timestamp'])

    # Combine thresholds (full-series) with in-window aggregations and earliest breach info
    df_pairs_duration = (
        df_trace_thresholds_duration
        .merge(agg_window, on=['cmdb_id','trace_name'], how='left')
        .merge(earliest, on=['cmdb_id','trace_name'], how='left')
    )

    # Fill NaNs for pairs without in-window data
    df_pairs_duration['in_window_count'] = df_pairs_duration['in_window_count'].fillna(0).astype(int)
    df_pairs_duration['breaches_count'] = df_pairs_duration['breaches_count'].fillna(0).astype(int)
    df_pairs_duration['max_value_in_window'] = df_pairs_duration['max_value_in_window'].astype(float)
    df_pairs_duration['earliest_breach_timestamp'] = df_pairs_duration['earliest_breach_timestamp'].fillna('')

    # 4) Top 10 pairs sorted by breaches_count desc then max_value_in_window desc
    top10_pairs = df_pairs_duration.sort_values(['breaches_count','max_value_in_window'], ascending=[False,False]).head(10)[[
        'cmdb_id','trace_name','full_series_count','p95_threshold',
        'in_window_count','breaches_count','earliest_breach_timestamp','max_value_in_window'
    ]].reset_index(drop=True)

    # 5) Per-minute tally: count unique (cmdb_id, trace_name) pairs breached at each timestamp within window
    if df_breaches_duration.empty:
        breach_tally_top10 = pd.DataFrame(columns=['timestamp','number_of_pairs_with_breach_at_that_timestamp'])
    else:
        df_breaches_duration['ts_str'] = df_breaches_duration['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
        breach_pairs_unique = df_breaches_duration[['ts_str','cmdb_id','trace_name']].drop_duplicates()
        breach_tally = (
            breach_pairs_unique.groupby('ts_str').size().reset_index(name='number_of_pairs_with_breach_at_that_timestamp')
        )
        breach_tally_top10 = breach_tally.sort_values('number_of_pairs_with_breach_at_that_timestamp', ascending=False).head(10).rename(columns={'ts_str':'timestamp'}).reset_index(drop=True)

# Final compact outputs (store intermediate tables in kernel as requested)
top10_pairs, breach_tally_top10

```
Out[11]:
```
```python
# Compose a compact plain-English summary from the previously computed results.
try:
    df_pairs_duration  # per-pair duration results
    top10_pairs
    breach_tally_top10
except NameError:
    summary = "Error: required intermediate tables not found in kernel."
else:
    total_pairs = int(df_pairs_duration.shape[0])
    pairs_with_breaches = int((df_pairs_duration['breaches_count'] > 0).sum())
    top_peak_ts = breach_tally_top10.at[0, 'timestamp'] if not breach_tally_top10.empty else 'N/A'
    top_peak_count = int(breach_tally_top10.at[0, 'number_of_pairs_with_breach_at_that_timestamp']) if not breach_tally_top10.empty else 0

    # Inspect pattern in the top10 pairs: how many have p95==0 and max_value_in_window==0
    p95_zero_and_zero_max = int(((top10_pairs['p95_threshold'] == 0) & (top10_pairs['max_value_in_window'] == 0)).sum())
    sample_pair_examples = top10_pairs[['cmdb_id','trace_name']].head(5).to_dict(orient='records')

    summary_lines = [
        "Trace duration metrics summary (window 2022-03-21 09:00–09:30 +0800):",
        "",
        f"- Total unique (cmdb_id, trace_name) duration pairs examined: {total_pairs}",
        f"- Pairs with >=1 in-window breach: {pairs_with_breaches}",
        "",
        "Top-10 breached pairs (shown separately) mostly exhibit this pattern:",
        f"  * Many top pairs have p95_threshold = 0 and max_value_in_window = 0, but were counted as breaches (value >= P95).",
        f"  * In the provided top-10, {p95_zero_and_zero_max} pairs match that zero-threshold/zero-value pattern.",
        f"  * Sample top pairs: {sample_pair_examples}",
        "",
        "Per-minute breach peaks (top 10 shown separately):",
        f"  * Highest concurrency of breached duration pairs at {top_peak_ts} with {top_peak_count} pairs breached.",
        "  * Other high-count minutes include 09:10 and 09:08 (see per-minute tally).",
        "",
        "Interpretation and recommendation:",
        "- Many duration series have global P95 == 0, and using the rule 'value >= P95' makes zero values count as breaches.",
        "- This produces widespread, likely spurious, breaches; adjust the rule for zero-baseline metrics (e.g., require value > P95 when P95==0,",
        "  use a small absolute threshold, or ignore metrics with persistent zeros) to reduce false positives.",
        "- Focus investigation on minutes around the peak (e.g., 09:11, 09:10, 09:08) and on duration series that have meaningful nonzero thresholds.",
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                  trace_name  full_series_count  p95_threshold  in_window_count  breaches_count  earliest_breach_timestamp  max_value_in_window
0      cartservice-0  trace.from_checkoutservice-1.duration_mean                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
1      cartservice-0   trace.from_checkoutservice-1.duration_p95                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
2      cartservice-2  trace.from_checkoutservice-1.duration_mean                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
3      cartservice-2   trace.from_checkoutservice-1.duration_p95                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
4  checkoutservice-1        trace.to_cartservice-0.duration_mean                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
5  checkoutservice-1         trace.to_cartservice-0.duration_p95                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
6  checkoutservice-1        trace.to_cartservice-2.duration_mean                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
7  checkoutservice-1         trace.to_cartservice-2.duration_p95                 25            0.0               25              25  2022-03-21 09:00:00 +0800                  0.0
8      cartservice-0  trace.from_checkoutservice-0.duration_mean                 24            0.0               24              24  2022-03-21 09:00:00 +0800                  0.0
9      cartservice-0   trace.from_checkoutservice-0.duration_p95                 24            0.0               24              24  2022-03-21 09:00:00 +0800                  0.0,                    timestamp  number_of_pairs_with_breach_at_that_timestamp
0  2022-03-21 09:11:00 +0800                                            208
1  2022-03-21 09:10:00 +0800                                            146
2  2022-03-21 09:08:00 +0800                                            115
3  2022-03-21 09:09:00 +0800                                            112
4  2022-03-21 09:07:00 +0800                                             74
5  2022-03-21 09:28:00 +0800                                             62
6  2022-03-21 09:29:00 +0800                                             60
7  2022-03-21 09:15:00 +0800                                             56
8  2022-03-21 09:06:00 +0800                                             56
9  2022-03-21 09:20:00 +0800                                             52)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Reuse existing df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'].astype('int64'), unit='s', utc=True)
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(tz)

# Window boundaries (inclusive)
start_sh = tz.localize(pd.Timestamp('2022-03-21 09:00:00'))
end_sh = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))

# 1) Select all trace_name containing ".row_count" (full series) to compute global P5 per pair
mask_rowcount_all = df_trace['trace_name'].str.contains(r'\.row_count', case=False, na=False)
df_row_all = df_trace[mask_rowcount_all].copy()

if df_row_all.empty:
    # No matching series -> empty outputs with required columns
    top20_rows = pd.DataFrame(columns=[
        'cmdb_id','trace_name','full_series_count','p5_threshold',
        'in_window_count','breaches_count','earliest_breach_timestamp','min_value_in_window'
    ])
    breach_tally_top10 = pd.DataFrame(columns=['timestamp','number_of_pairs_with_breach_at_that_timestamp'])
else:
    # Compute global P5 (full-series) per (cmdb_id, trace_name)
    df_thresholds_rows = (
        df_row_all
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(full_series_count='count', p5_threshold=lambda x: float(x.quantile(0.05)))
    )

    # 2) Filter trace rows to the incident window and containing ".row_count"
    mask_window = (df_trace['ts_sh'] >= start_sh) & (df_trace['ts_sh'] <= end_sh) & mask_rowcount_all
    df_window_rows = df_trace[mask_window].copy()

    # Merge thresholds into window rows
    df_window_merged_rows = df_window_rows.merge(df_thresholds_rows, on=['cmdb_id','trace_name'], how='left')

    # 3) Determine breaches: value <= p5_threshold (if p5_threshold is NaN -> not a breach)
    def is_breach_row(r):
        thr = r['p5_threshold']
        val = r['value']
        if pd.isna(thr):
            return False
        return val <= thr

    df_window_merged_rows['is_breach'] = df_window_merged_rows.apply(is_breach_row, axis=1)

    # Aggregations per pair
    agg_window = df_window_merged_rows.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        in_window_count=('value','count'),
        breaches_count=('is_breach','sum'),
        min_value_in_window=('value','min')
    )

    # Earliest breach timestamp per pair (if any)
    df_breaches_rows = df_window_merged_rows[df_window_merged_rows['is_breach']].copy()
    if not df_breaches_rows.empty:
        earliest = (
            df_breaches_rows.groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(earliest_breach_ts=('ts_sh','min'))
        )
        earliest['earliest_breach_timestamp'] = earliest['earliest_breach_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
        earliest = earliest.drop(columns=['earliest_breach_ts'])
    else:
        earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_breach_timestamp'])

    # Combine thresholds with in-window aggregates and earliest breach info
    df_result_rows = (
        df_thresholds_rows
        .merge(agg_window, on=['cmdb_id','trace_name'], how='left')
        .merge(earliest, on=['cmdb_id','trace_name'], how='left')
    )

    # Fill NaNs for pairs without in-window data
    df_result_rows['in_window_count'] = df_result_rows['in_window_count'].fillna(0).astype(int)
    df_result_rows['breaches_count'] = df_result_rows['breaches_count'].fillna(0).astype(int)
    df_result_rows['min_value_in_window'] = df_result_rows['min_value_in_window'].astype(float)
    df_result_rows['earliest_breach_timestamp'] = df_result_rows['earliest_breach_timestamp'].fillna('')

    # 4) Top 20 pairs sorted by breaches_count desc then min_value_in_window asc
    top20_rows = df_result_rows.sort_values(['breaches_count','min_value_in_window'], ascending=[False,True]).head(20)[[
        'cmdb_id','trace_name','full_series_count','p5_threshold',
        'in_window_count','breaches_count','earliest_breach_timestamp','min_value_in_window'
    ]].reset_index(drop=True)

    # 5) Per-minute tally: number_of_pairs_with_breach_at_that_timestamp (unique (cmdb_id,trace_name) per timestamp)
    if df_breaches_rows.empty:
        breach_tally_top10 = pd.DataFrame(columns=['timestamp','number_of_pairs_with_breach_at_that_timestamp'])
    else:
        df_breaches_rows['ts_str'] = df_breaches_rows['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %z')
        breach_pairs_unique = df_breaches_rows[['ts_str','cmdb_id','trace_name']].drop_duplicates()
        breach_tally = (
            breach_pairs_unique.groupby('ts_str').size().reset_index(name='number_of_pairs_with_breach_at_that_timestamp')
        )
        breach_tally_top10 = breach_tally.sort_values('number_of_pairs_with_breach_at_that_timestamp', ascending=False).head(10).rename(columns={'ts_str':'timestamp'}).reset_index(drop=True)

# Final compact outputs (top20_rows and breach_tally_top10)
top20_rows, breach_tally_top10

```
Out[12]:
```
Summary (plain English):

- Top row_count pairs (window 2022-03-21 09:00–09:30 +08):
  - Many trace .row_count series show widespread breaches: their full-series P5 = 1.0 and every minute in-window is ≤ P5, so breaches_count is large (often equal to in_window_count).
  - Examples from the top pairs (all have p5_threshold = 1.0 and min_value_in_window = 1.0):
    - checkoutservice-2 — trace.from_frontend-0.row_count: full_series_count=20, in_window_count=20, breaches_count=20, earliest breach 2022-03-21 09:00:00 +0800
    - checkoutservice-2 — trace.from_frontend-1.row_count: full_series_count=20, in_window_count=20, breaches_count=20, earliest 09:00
    - frontend-0 / frontend-1 — trace.to_checkoutservice-2.row_count: full_series_count=20, in_window_count=20, breaches_count=20, earliest 09:00
    - Multiple other checkout/frontend/cart/productcatalog/shipping/payment related row_count traces appear in the top 20 with similar patterns (see table).
  - Interpretation: these are low-count row metrics (P5 = 1) and the window values equal that low baseline, producing many breaches by the rule value ≤ P5.

- Per-minute breach concurrency (top 10 minutes):
  - 2022-03-21 09:11:00 +0800 — 196 pairs
  - 2022-03-21 09:10:00 +0800 — 139 pairs
  - 2022-03-21 09:07:00 +0800 — 88 pairs
  - 2022-03-21 09:08:00 +0800 — 86 pairs
  - 2022-03-21 09:01:00 +0800 — 86 pairs
  - 2022-03-21 09:09:00 +0800 — 85 pairs
  - 2022-03-21 09:23:00 +0800 — 84 pairs
  - 2022-03-21 09:27:00 +0800 — 80 pairs
  - 2022-03-21 09:16:00 +0800 — 80 pairs
  - 2022-03-21 09:22:00 +0800 — 80 pairs

- Conclusion and recommendations:
  - Many row_count breaches are likely artifacts of the chosen rule and low baselines (P5 = 1). A value of 1 is common and triggers breaches even though it may be normal.
  - Tune detection for row_count metrics: use a different rule for sparse/low-count metrics (e.g., require sustained drops, use absolute thresholds >0, or use relative change), or exclude row_count series with P5 ≤ 1 from immediate alerting.
  - Investigate the high-concurrency minutes (especially 09:10–09:11) to see if there was a real service-level event affecting many request flows (checkout/frontend/cart interactions), but prioritize metrics with meaningful nonzero baselines first.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                  trace_name  full_series_count  p5_threshold  in_window_count  breaches_count  earliest_breach_timestamp  min_value_in_window
0         checkoutservice-2             trace.from_frontend-0.row_count                 20           1.0               20              20  2022-03-21 09:00:00 +0800                  1.0
1         checkoutservice-2             trace.from_frontend-1.row_count                 20           1.0               20              20  2022-03-21 09:00:00 +0800                  1.0
2                frontend-0        trace.to_checkoutservice-2.row_count                 20           1.0               20              20  2022-03-21 09:00:00 +0800                  1.0
3                frontend-1        trace.to_checkoutservice-2.row_count                 20           1.0               20              20  2022-03-21 09:00:00 +0800                  1.0
4         checkoutservice-0             trace.from_frontend-0.row_count                 19           1.0               19              19  2022-03-21 09:01:00 +0800                  1.0
5         checkoutservice-1             trace.from_frontend-1.row_count                 20           1.0               20              19  2022-03-21 09:02:00 +0800                  1.0
6                frontend-0        trace.to_checkoutservice-0.row_count                 19           1.0               19              19  2022-03-21 09:01:00 +0800                  1.0
7                frontend-1        trace.to_checkoutservice-1.row_count                 20           1.0               20              19  2022-03-21 09:02:00 +0800                  1.0
8         checkoutservice-1             trace.from_frontend-0.row_count                 19           1.0               19              18  2022-03-21 09:00:00 +0800                  1.0
9         checkoutservice-1  trace.to_productcatalogservice-2.row_count                 25           1.0               25              18  2022-03-21 09:01:00 +0800                  1.0
10        checkoutservice-2        trace.to_shippingservice-1.row_count                 25           1.0               25              18  2022-03-21 09:00:00 +0800                  1.0
11               frontend-0        trace.to_checkoutservice-1.row_count                 19           1.0               19              18  2022-03-21 09:00:00 +0800                  1.0
12  productcatalogservice-2      trace.from_checkoutservice-1.row_count                 25           1.0               25              18  2022-03-21 09:01:00 +0800                  1.0
13        shippingservice-1      trace.from_checkoutservice-2.row_count                 25           1.0               25              18  2022-03-21 09:00:00 +0800                  1.0
14            cartservice-0      trace.from_checkoutservice-1.row_count                 25           1.0               25              17  2022-03-21 09:01:00 +0800                  1.0
15            cartservice-2      trace.from_checkoutservice-1.row_count                 25           1.0               25              17  2022-03-21 09:01:00 +0800                  1.0
16        checkoutservice-0        trace.to_shippingservice-0.row_count                 24           1.0               24              17  2022-03-21 09:01:00 +0800                  1.0
17        checkoutservice-1            trace.to_cartservice-0.row_count                 25           1.0               25              17  2022-03-21 09:01:00 +0800                  1.0
18        checkoutservice-1            trace.to_cartservice-2.row_count                 25           1.0               25              17  2022-03-21 09:01:00 +0800                  1.0
19        checkoutservice-1         trace.to_paymentservice-1.row_count                 17           1.0               17              17  2022-03-21 09:00:00 +0800                  1.0,                    timestamp  number_of_pairs_with_breach_at_that_timestamp
0  2022-03-21 09:11:00 +0800                                            196
1  2022-03-21 09:10:00 +0800                                            139
2  2022-03-21 09:07:00 +0800                                             88
3  2022-03-21 09:08:00 +0800                                             86
4  2022-03-21 09:01:00 +0800                                             86
5  2022-03-21 09:09:00 +0800                                             85
6  2022-03-21 09:23:00 +0800                                             84
7  2022-03-21 09:27:00 +0800                                             80
8  2022-03-21 09:16:00 +0800                                             80
9  2022-03-21 09:22:00 +0800                                             80)```
```